 # Multi-Domain Recommendation Data Processing



 This notebook processes multi-domain recommendation data with overlap handling and benchmark generation.

In [ ]:
import os
import random
from typing import Dict, List, Set, Tuple

import pandas as pd
from recbole.utils import init_seed


 ## Configuration

In [ ]:
# Constants
SEED = 2025
MAX_SEQUENCE_LENGTH = 128
CORE_THRESHOLDS = {"user": 3, "item": 3}

# Dataset configuration
DATA_DIR = "./raw"
DATASETS = ["Electronics", "Sports", "Tools", "Toys"]

# Initialize random seed
init_seed(2025, True)

# Initialize dataset prefix and create output directory
dataset_prefix = "".join([d[0] for d in DATASETS])
os.makedirs(dataset_prefix, exist_ok=True)


 ## Dataset Name Mapping

In [ ]:
dataset2fullname = {
    "Appliances": "Amazon_Appliances",
    "Arts": "Amazon_Arts_Crafts_and_Sewing",
    "Automotive": "Amazon_Automotive",
    "Beauty": "Amazon_All_Beauty",
    "Books": "Amazon_Books",
    "CDs": "Amazon_CDs_and_Vinyl",
    "Cell": "Amazon_Cell_Phones_and_Accessories",
    "Clothing": "Amazon_Clothing_Shoes_and_Jewelry",
    "Electronics": "Amazon_Electronics",
    "epinions": "epinions",
    "Fashion": "Amazon_AMAZON_FASHION",
    "food": "Food",
    "Food": "Amazon_Grocery_and_Gourmet_Food",
    "Garden": "Amazon_Patio_Lawn_and_Garden",
    "Gift": "Amazon_Gift_Cards",
    "Video": "Amazon_Video_Games",
    "Instruments": "Amazon_Musical_Instruments",
    "Kindle": "Amazon_Kindle_Store",
    "Kitchen": "Amazon_Home_and_Kitchen",
    "Luxury": "Amazon_Luxury_Beauty",
    "Magazine": "Amazon_Magazine_Subscriptions",
    "ml100k": "ml-100k",
    "Movies": "Amazon_Movies_and_TV",
    "Music": "Amazon_Digital_Music",
    "Office": "Amazon_Office_Products",
    "Pantry": "Amazon_Prime_Pantry",
    "Pet": "Amazon_Pet_Supplies",
    "Scientific": "Amazon_Industrial_and_Scientific",
    "Software": "Amazon_Software",
    "Sports": "Amazon_Sports_and_Outdoors",
    "Tools": "Amazon_Tools_and_Home_Improvement",
    "Toys": "Amazon_Toys_and_Games",
}


 ## Data Loading Utilities

In [ ]:
def load_ratings(file_path: str) -> Dict:
    """Load and preprocess ratings data from a file."""
    df = pd.read_csv(
        file_path,
        sep="\t",
        header=0,
        names=["user", "item", "rating", "time"],
        dtype={"user": str, "item": str, "rating": float, "time": int},
    )
    df = df.drop_duplicates().reset_index(drop=True)
    return {
        "users": set(df["user"].unique()),
        "items": set(df["item"].unique()),
        "df": df[["user", "item", "rating", "time"]],
    }


# Load raw data
rating_paths = {d: os.path.join(DATA_DIR, dataset2fullname[d], f"{dataset2fullname[d]}.inter") for d in DATASETS}
domain_data = {d: load_ratings(rating_paths[d]) for d in DATASETS}
print("Data loaded.")


 ## Core Filtering Functions

In [ ]:
def kcore_filter(df: pd.DataFrame, user_thresh: int = 5, item_thresh: int = 5) -> pd.DataFrame:
    """Apply k-core filtering to ensure minimum user/item interactions."""
    while True:
        user_counts = df["user"].value_counts()
        item_counts = df["item"].value_counts()

        user_mask = df["user"].isin(user_counts[user_counts >= user_thresh].index)
        item_mask = df["item"].isin(item_counts[item_counts >= item_thresh].index)
        combined_mask = user_mask & item_mask

        if combined_mask.all():
            break
        df = df[combined_mask]
    return df


def align_domains(data_dict: Dict, user_thresh: int = 4, item_thresh: int = 4) -> Tuple[Set, Dict]:
    """Align multiple domains to find common users with sufficient interactions."""
    common_users = set.intersection(*[d["users"] for d in data_dict.values()])

    while True:
        prev_size = len(common_users)
        print(prev_size)
        filtered_dfs = {}

        for domain, data in data_dict.items():
            filtered_df = data["df"][data["df"]["user"].isin(common_users)]
            filtered_df = kcore_filter(filtered_df, user_thresh, item_thresh)
            filtered_dfs[domain] = filtered_df

        new_common = set.intersection(*[set(df["user"].unique()) for df in filtered_dfs.values()])

        if len(new_common) == prev_size:
            final_dfs = {domain: kcore_filter(df, user_thresh, item_thresh) for domain, df in filtered_dfs.items()}
            return new_common, final_dfs

        common_users = new_common


# Find overlapping users and their data
overlapped_users, overlapped_dfs = align_domains(
    domain_data, user_thresh=CORE_THRESHOLDS["user"], item_thresh=CORE_THRESHOLDS["item"]
)


 ## Data Transformation

In [ ]:
def add_domain_suffix(dfs: Dict[str, pd.DataFrame], datasets: List[str]) -> Dict[str, pd.DataFrame]:
    """Add domain-specific suffixes to item IDs."""
    suffixed_dfs = {}
    for idx, domain in enumerate(datasets):
        if domain not in dfs:
            continue
        df = dfs[domain].copy()
        suffix = chr(65 + idx)  # A, B, C, etc.
        df["item"] += f"_{suffix}"
        df["domain"] = domain
        suffixed_dfs[domain] = df
    return suffixed_dfs


# Process overlapping data
suffixed_dfs = add_domain_suffix(overlapped_dfs, DATASETS)
combined_df = pd.concat(suffixed_dfs.values())
combined_df = combined_df.sort_values(by=["user", "time"]).groupby("user", group_keys=False).apply(lambda x: x.tail(MAX_SEQUENCE_LENGTH))


def split_combined_df_strict(
    combined_df: pd.DataFrame,
    datasets: List[str],
) -> Dict[str, pd.DataFrame]:
    """Split combined DataFrame and ensure same format as original suffixed_dfs."""
    split_dfs = {}

    for domain in datasets:
        domain_df = combined_df[combined_df["domain"] == domain].copy()
        if not domain_df.empty:
            split_dfs[domain] = {}
            split_dfs[domain]["df"] = domain_df
            split_dfs[domain]["users"] = set(domain_df["user"].unique())
            split_dfs[domain]["items"] = set(domain_df["item"].unique())

    return split_dfs


split_dfs = split_combined_df_strict(combined_df, DATASETS)


In [ ]:
new_overlapped_users, new_overlapped_dfs = align_domains(
    split_dfs, user_thresh=CORE_THRESHOLDS["user"], item_thresh=CORE_THRESHOLDS["item"]
)

new_combined_df = pd.concat(new_overlapped_dfs.values())
new_combined_df = new_combined_df.sort_values(by=["user", "time"]).groupby("user", group_keys=False).apply(lambda x: x.tail(MAX_SEQUENCE_LENGTH))


 ## Output Generation

In [ ]:
def write_items(items: List, path: str) -> None:
    """Write item list to file."""
    pd.Series(items).to_csv(path, index=False, header=False)
    print(f"Items written to {path}")


# Generate item and user mappings
overlapped_items = sorted(new_combined_df["item"].unique(), key=lambda x: x.split("_")[-1])
item2index = {item: idx + 1 for idx, item in enumerate(overlapped_items)}
user2index = {user: idx + 1 for idx, user in enumerate(new_overlapped_users)}

# Write item list
write_items(overlapped_items, os.path.join(dataset_prefix, f"{dataset_prefix}.item"))


 ## Benchmark Generation

In [ ]:
def generate_and_save_benchmark(
    combined_df: pd.DataFrame, user2index: Dict, item2index: Dict, dataset_prefix: str
):
    combined_df = combined_df.copy()
    combined_df["user"] = combined_df["user"].map(user2index)
    combined_df["item"] = combined_df["item"].map(item2index)

    user2items = combined_df.groupby("user")["item"].agg(list).to_dict()

    def make_rows(user, seq, mode):
        if len(seq) >= 4:
            if mode == "train":
                return (user, seq[:-3], seq[-3])
            elif mode == "valid":
                return (user, seq[:-2], seq[-2])
            elif mode == "test":
                return (user, seq[:-1], seq[-1])
        elif len(seq) == 3:
            if mode == "train":
                return (user, seq[:-2], seq[-2])
            elif mode == "test":
                return (user, seq[:-1], seq[-1])
        elif len(seq) == 2:
            if mode == "test":
                return (user, seq[:-1], seq[-1])
        return None

    results = {m: [] for m in ["train", "valid", "test"]}

    for user, seq in user2items.items():
        for mode in ["train", "valid", "test"]:
            row = make_rows(user, seq, mode)
            if row:
                u, history, target = row
                results[mode].append((u, " ".join(map(str, history)), target))

    for mode, rows in results.items():
        if not rows:
            continue
        df = pd.DataFrame(
            rows, columns=["user_id:token", "item_id_list:token_seq", "item_id:token"]
        )
        df.sort_values("user_id:token", inplace=True)
        df.to_csv(
            f"{dataset_prefix}/{dataset_prefix}.full.{mode}.inter",
            sep="\t",
            index=False,
        )
        print(f"Generated {mode}: {len(df)} rows")


generate_and_save_benchmark(
    combined_df=new_combined_df,
    user2index=user2index,
    item2index=item2index,
    dataset_prefix=dataset_prefix,
)


 ## Domain-Specific Splits

In [ ]:
def get_domain_ranges(
    items: List[str], datasets: List[str]
) -> Dict[str, Tuple[int, int]]:
    """Calculate index ranges for each domain's items."""
    domain_ranges = {}
    current_index = 1  # Using 1-based indexing

    for domain_idx, domain in enumerate(datasets):
        domain_suffix = f"_{chr(65 + domain_idx)}"  # _A, _B, _C, etc.
        domain_items = [item for item in items if item.endswith(domain_suffix)]
        item_count = len(domain_items)

        domain_ranges[domain] = (current_index, current_index + item_count - 1)
        current_index += item_count

    return domain_ranges


def create_domain_specific_datasets(
    full_df: pd.DataFrame,
    domain_name: str,
    min_idx: int,
    max_idx: int,
    dataset_prefix: str,
) -> None:
    """Create domain-specific train/valid/test datasets from the full dataset."""
    print(f"Processing domain {domain_name} (items {min_idx}-{max_idx})")

    # Initialize containers for each split
    domain_data = {"train": [], "valid": [], "test": []}

    for _, row in full_df.iterrows():
        # Convert item sequences to integers
        user_id = row["user_id:token"]
        
        tst_seq = list(map(int, row["item_id_list:token_seq"].split()))
        tst_seq.append(row["item_id:token"])
        val_seq = tst_seq[:-1]
        trn_seq = tst_seq[:-2]

        if len(tst_seq) >= 2 and min_idx <= tst_seq[-1] <= max_idx:
            tst_seq_dom = [item for item in tst_seq if min_idx <= item <= max_idx]
            if len(tst_seq_dom) >= 2:
                tst_sample = [
                    user_id,
                    " ".join(map(str, tst_seq_dom[:-1])),
                    tst_seq_dom[-1],
                ]
                domain_data["test"].append(tst_sample)

        if len(val_seq) >= 2 and min_idx <= val_seq[-1] <= max_idx:
            val_seq_dom = [item for item in val_seq if min_idx <= item <= max_idx]
            if len(val_seq_dom) >= 2:
                val_sample = [
                    user_id,
                    " ".join(map(str, val_seq_dom[:-1])),
                    val_seq_dom[-1],
                ]
                domain_data["valid"].append(val_sample)

        if len(trn_seq) >= 2:
            trn_seq_dom = [item for item in trn_seq if min_idx <= item <= max_idx]
            if len(trn_seq_dom) >= 2:
                trn_sample = [
                    user_id,
                    " ".join(map(str, trn_seq_dom[:-1])),
                    trn_seq_dom[-1],
                ]
                domain_data["train"].append(trn_sample)

    for split_type, rows in domain_data.items():
        if rows:
            output_df = pd.DataFrame(
                rows,
                columns=["user_id:token", "item_id_list:token_seq", "item_id:token"],
            )
            output_df.sort_values("user_id:token").to_csv(
                f"{dataset_prefix}/{dataset_prefix}.{domain_name}.{split_type}.inter",
                sep="\t",
                index=False,
            )


In [ ]:
# Calculate domain ranges
domain_ranges = get_domain_ranges(overlapped_items, DATASETS)

# Load the full test dataset
test_df = pd.read_csv(f"{dataset_prefix}/{dataset_prefix}.full.test.inter", sep="\t")

# Process each domain
for domain_idx, domain in enumerate(DATASETS):
    domain_key = f"dom{domain_idx + 1}"
    min_index, max_index = domain_ranges[domain]

    create_domain_specific_datasets(
        full_df=test_df, domain_name=domain_key, min_idx=min_index, max_idx=max_index, dataset_prefix=dataset_prefix
    )



<br/>
<br/>
<br/>
<br/>
<br/>
<br/>
<br/>
<br/>
<br/>
<br/>
<br/>
<br/>
<br/>
<br/>
<br/>
<br/>